# Simon's Algorithm in PennyLane (3-Bit Hidden-Period Oracle)

This notebook implements **Simon's algorithm** using PennyLane for a $3$-bit hidden-period oracle $U_f$ satisfying $f(x) = f(y) \iff y = x \oplus s$ for a secret bitstring $s \in \{0, 1\}^3$.

**Workflow:**
1. Construct a $2$-to-$1$ quantum oracle $U_f$ using `CNOT` gates.
2. Sample the PennyLane circuit to obtain bitstrings $y$ satisfying $y \cdot s = 0 \pmod 2$.
3. Recover the hidden period $s$ via Gaussian elimination over $GF(2)$.
4. Verify recovery across all non-zero $3$-bit secrets.
5. Benchmark quantum oracle query counts against classical brute-force collision search.

## 1. Setup & Imports

In [ ]:
# Run once if PennyLane is not installed:
# !pip install pennylane numpy matplotlib

import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 2. Constructing the 3-Bit Simon Oracle

For a secret string $s \in \{0, 1\}^n$, the oracle $U_f$ maps $|x\rangle|0\rangle \mapsto |x\rangle|f(x)\rangle$ such that $f(x) = f(y) \iff y = x \oplus s$. To build a $2$-to-$1$ oracle in gates:
1. Copy the input register ($0, 1, 2$) to the output register ($3, 4, 5$) using `CNOT` gates.
2. Find the first bit index $j$ where $s_j = 1$. For every index $i$ where $s_i = 1$, apply a `CNOT` from input qubit $j$ to output qubit $n + i$.

In [ ]:
def simon_oracle(secret: str):
    """Applies a 2-to-1 Simon oracle U_f for a binary secret string."""
    n = len(secret)
    
    # Step 1: Copy input register to output register
    for i in range(n):
        qml.CNOT(wires=[i, n + i])
        
    # Step 2: Enforce f(x) = f(x XOR s) if s != 000
    if "1" in secret:
        j = secret.index("1")  # First non-zero bit index
        for i, bit in enumerate(secret):
            if bit == "1":
                qml.CNOT(wires=[j, n + i])

## 3. PennyLane Circuit & Sampling

The circuit applies Hadamard gates to the $3$-qubit input register, queries `simon_oracle`, and applies a second layer of Hadamards to the input register. Measuring the input register yields a bitstring $y$ guaranteed to satisfy $y \cdot s = 0 \pmod 2$.

In [ ]:
SECRET = "101"
N = len(SECRET)

# Single-shot device to count exact oracle queries
dev_single = qml.device("default.qubit", wires=2 * N, shots=1)

@qml.qnode(dev_single)
def simon_single_query(secret: str):
    for i in range(N):
        qml.Hadamard(wires=i)
    
    simon_oracle(secret)
    
    for i in range(N):
        qml.Hadamard(wires=i)
        
    return qml.sample(wires=range(N))

# Visualize the circuit
print(qml.draw(simon_single_query)(SECRET))

## 4. Solving Linear Equations over $GF(2)$

To recover $s$, we sample until we collect $n - 1 = 2$ linearly independent non-zero bitstrings $y_k$, then compute the $1$-dimensional nullspace of the matrix $Y$ over $GF(2)$ using Gaussian elimination.

In [ ]:
def gf2_rref(matrix: np.ndarray):
    """Returns the Reduced Row Echelon Form of a binary matrix over GF(2) and its pivot columns."""
    M = matrix.copy() % 2
    rows, cols = M.shape
    pivots = []
    r = 0

    for c in range(cols):
        if r >= rows:
            break
        # Find a pivot in column c at or below row r
        pivot_row = None
        for i in range(r, rows):
            if M[i, c] == 1:
                pivot_row = i
                break
        if pivot_row is None:
            continue

        # Swap current row with pivot row
        M[[r, pivot_row]] = M[[pivot_row, r]]
        
        # Eliminate all other 1s in column c
        for i in range(rows):
            if i != r and M[i, c] == 1:
                M[i] = (M[i] ^ M[r]) % 2

        pivots.append(c)
        r += 1

    return M, pivots

def solve_simon_gf2(equations: list[np.ndarray]) -> str:
    """Finds the non-trivial secret string s satisfying Y * s = 0 (mod 2)."""
    M, pivots = gf2_rref(np.array(equations, dtype=int))
    n = M.shape[1]
    free_cols = [c for c in range(n) if c not in pivots]
    
    if len(free_cols) != 1:
        raise ValueError("Expected exactly n-1 linearly independent equations.")
    
    free_col = free_cols[0]
    s = np.zeros(n, dtype=int)
    s[free_col] = 1
    
    for row_idx, pivot_col in enumerate(pivots):
        s[pivot_col] = M[row_idx, free_col]
        
    return "".join(map(str, s))

def run_simons_algorithm(secret: str):
    """Samples the PennyLane circuit until n-1 independent equations are found."""
    n = len(secret)
    independent_eqs = []
    queries = 0

    while len(independent_eqs) < n - 1:
        sample = np.array(simon_single_query(secret), dtype=int).flatten()
        queries += 1
        
        if np.all(sample == 0):
            continue
            
        candidate = independent_eqs + [sample]
        _, pivots = gf2_rref(np.array(candidate, dtype=int))
        
        if len(pivots) > len(independent_eqs):
            independent_eqs.append(sample)

    recovered = solve_simon_gf2(independent_eqs)
    return recovered, queries, independent_eqs

recovered_s, q_count, eqs = run_simons_algorithm(SECRET)
print(f"True secret:        {SECRET}")
print(f"Recovered secret:   {recovered_s}")
print(f"Oracle queries:     {q_count}")
print(f"Measured equations: {[ ''.join(map(str, e)) for e in eqs ]}")

## 5. Verification Across All Non-Zero 3-Bit Secrets

In [ ]:
all_secrets = [f"{i:03b}" for i in range(1, 2**N)]

print(f"{'Secret':<10} | {'Recovered':<10} | {'Match':<8} | {'Queries'}")
print("-" * 45)
for s in all_secrets:
    rec, queries, _ = run_simons_algorithm(s)
    assert rec == s, f"Mismatch for {s}: got {rec}"
    print(f"{s:<10} | {rec:<10} | {str(rec == s):<8} | {queries}")

print("\nAll 7 non-zero 3-bit hidden periods verified successfully!")

## 6. Query Count Comparison (Quantum vs. Classical Brute-Force)

A classical search queries distinct random inputs until it finds a collision $f(x_1) = f(x_2)$, yielding $s = x_1 \oplus x_2$. For $n = 3$, classical worst-case deterministic search needs $2^{n-1} + 1 = 5$ queries, while randomized search scales as $O(2^{n/2})$ compared to Simon's $O(n)$.

In [ ]:
def classical_oracle_eval(x: int, secret_int: int) -> int:
    """Classical 2-to-1 function satisfying f(x) = f(x XOR s)."""
    return min(x, x ^ secret_int)

def classical_brute_force(secret: str):
    """Queries random distinct inputs until a collision f(x1) == f(x2) is found."""
    n = len(secret)
    secret_int = int(secret, 2)
    inputs = np.random.permutation(2**n)
    seen_outputs = {}
    
    for queries, x in enumerate(inputs, start=1):
        y = classical_oracle_eval(int(x), secret_int)
        if y in seen_outputs:
            recovered_int = int(x) ^ seen_outputs[y]
            return f"{recovered_int:0{n}b}", queries
        seen_outputs[y] = int(x)

# Benchmark over 200 trials
TRIALS = 200
quantum_queries = [run_simons_algorithm(SECRET)[1] for _ in range(TRIALS)]
classical_queries = [classical_brute_force(SECRET)[1] for _ in range(TRIALS)]

print(f"Quantum average queries (n={N}):   {np.mean(quantum_queries):.2f}")
print(f"Classical average queries (n={N}): {np.mean(classical_queries):.2f}")

# Plot comparison
plt.figure(figsize=(8, 4.5))
bins = np.arange(1, max(max(quantum_queries), max(classical_queries)) + 2) - 0.5
plt.hist(quantum_queries, bins=bins, alpha=0.7, label=f"Simon's Algorithm (mean={np.mean(quantum_queries):.2f})", color="#4C72B0", edgecolor="black")
plt.hist(classical_queries, bins=bins, alpha=0.6, label=f"Classical Brute-Force (mean={np.mean(classical_queries):.2f})", color="#DD8452", edgecolor="black")

plt.xlabel("Number of Oracle Queries")
plt.ylabel(f"Frequency (out of {TRIALS} trials)")
plt.title(f"Query Count Comparison for 3-Bit Simon's Problem (s = {SECRET})")
plt.xticks(range(1, int(bins[-1]) + 1))
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()